# BRFSS 2015 Diabetes Health Indicators — A Study

**Dataset:** `diabetes_012_health_indicators_BRFSS2015.csv` (~253,680 rows)
from the CDC's Behavioral Risk Factor Surveillance System 2015 survey
([Kaggle](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset)).

**Target — `Diabetes_012`:**
- `0` = no diabetes
- `1` = prediabetes
- `2` = diabetes

Unlike the synthetic 100k dataset, this one is built from a **real** national
health survey, so every column is a survey answer already coded as a number.

Run cells with **Shift + Enter**.

## 1. Setup

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("pandas:", pd.__version__)

## 2. Load the dataset

Uses a local copy if present, otherwise downloads via `kagglehub` (cached after the first run).

In [ ]:
CSV_NAME = "diabetes_012_health_indicators_BRFSS2015.csv"


def load_dataset():
    """Load the 012 CSV locally if present, else download via kagglehub."""
    if os.path.exists(CSV_NAME):
        print("Loading local file:", CSV_NAME)
        return pd.read_csv(CSV_NAME)

    import kagglehub
    print("No local CSV found. Downloading via kagglehub...")
    path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")

    matches = glob.glob(os.path.join(path, "**", CSV_NAME), recursive=True)
    if not matches:  # fall back to any csv with '012' in the name
        matches = [f for f in glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
                   if "012" in os.path.basename(f)]
    if not matches:
        raise FileNotFoundError(f"Could not find {CSV_NAME} in {path}")

    print("Loading:", matches[0])
    return pd.read_csv(matches[0])


df = load_dataset()
df.head()

## 3. How big is it?

In [ ]:
rows, cols = df.shape
print(f"Rows    : {rows:,}")
print(f"Columns : {cols}")
print(f"Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumns:", list(df.columns))

## 4. Data types, missing values, duplicates

BRFSS is pre-cleaned, so we mostly expect zero missing values — but duplicates are common because many people give identical answers.

In [ ]:
df.info()

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isnull().sum(),
    "unique": df.nunique(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
})
print(f"Exact duplicate rows: {df.duplicated().sum():,}"
      f" ({df.duplicated().mean()*100:.1f}%)\n")
summary

## 5. Summary statistics

In [ ]:
df.describe().T

## 6. Class balance — the headline problem

Three classes, and they are **severely imbalanced**. Prediabetes (1) is tiny,
which makes it the hardest class to predict.

In [ ]:
target = "Diabetes_012"
labels = {0: "No diabetes", 1: "Prediabetes", 2: "Diabetes"}

counts = df[target].value_counts().sort_index()
pcts = df[target].value_counts(normalize=True).sort_index() * 100
balance = pd.DataFrame({"count": counts, "percent": pcts.round(2)})
balance.index = [labels[i] for i in balance.index]
print(balance)
print(f"\nAlways predicting 'No diabetes' would score {pcts.iloc[0]:.1f}% accuracy.")

plt.figure(figsize=(6, 4))
ax = sns.barplot(x=[labels[i] for i in counts.index], y=counts.values,
                 hue=[labels[i] for i in counts.index], palette="Set2", legend=False)
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.title("Class balance (Diabetes_012)")
plt.ylabel("count")
plt.show()

## 7. Correlation heatmap

Every column is numeric, so we can correlate all of them at once. Note the
`0/1/2` target is ordinal, so these are treated as ordered here.

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(15, 13))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, annot_kws={"size": 7},
            cbar_kws={"shrink": 0.8})
plt.title("Correlation heatmap — BRFSS 2015 diabetes indicators")
plt.tight_layout()
plt.savefig("brfss_correlation_heatmap.png", dpi=150)
plt.show()
print("Saved to brfss_correlation_heatmap.png")

## 8. What correlates most with the diabetes target?

In [ ]:
target_corr = corr[target].drop(target).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
sns.barplot(x=target_corr.values, y=target_corr.index, hue=target_corr.index,
            palette="coolwarm_r", legend=False)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation of each feature with Diabetes_012")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

target_corr.round(3)

## 9. Diabetes prevalence across the strongest risk factors

Correlation is abstract. This is more intuitive: for each level of a risk
factor, what fraction of people actually *have* diabetes (class 2)?

In [ ]:
df["is_diabetic"] = (df[target] == 2).astype(int)

def prevalence_by(col):
    g = df.groupby(col)["is_diabetic"].mean() * 100
    return g.round(1)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# General health (1=excellent ... 5=poor)
prevalence_by("GenHlth").plot(kind="bar", ax=axes[0, 0], color="#d1495b")
axes[0, 0].set_title("Diabetes % by General Health (1=excellent, 5=poor)")
axes[0, 0].set_ylabel("% diabetic")

# High blood pressure
prevalence_by("HighBP").plot(kind="bar", ax=axes[0, 1], color="#edae49")
axes[0, 1].set_title("Diabetes % by HighBP (0=no, 1=yes)")

# Age bucket (BRFSS 1..13)
prevalence_by("Age").plot(kind="bar", ax=axes[1, 0], color="#00798c")
axes[1, 0].set_title("Diabetes % by Age bucket (1=18-24 ... 13=80+)")
axes[1, 0].set_ylabel("% diabetic")

# Difficulty walking
prevalence_by("DiffWalk").plot(kind="bar", ax=axes[1, 1], color="#66a182")
axes[1, 1].set_title("Diabetes % by DiffWalk (0=no, 1=yes)")

plt.tight_layout()
plt.savefig("brfss_prevalence.png", dpi=150)
plt.show()

## 10. BMI distribution by diabetes status

In [ ]:
plt.figure(figsize=(9, 5))
sns.kdeplot(data=df, x="BMI", hue=df[target].map(labels),
            fill=True, common_norm=False, palette="Set2", clip=(10, 60))
plt.title("BMI distribution by diabetes status")
plt.xlim(10, 60)
plt.show()

print(df.groupby(target)["BMI"].median().rename(labels).round(1))

## 11. A note on modelling this data

If you go on to build a classifier:

- **Collapse to binary if you only care about diabetes.** Prediabetes (class 1)
  is <2% of rows and very hard to separate — many studies merge `1` and `2`
  into a single "at risk" class, or drop `1` entirely.
- **Handle the imbalance** with a stratified split plus `class_weight="balanced"`
  (or `scale_pos_weight` for boosting), and judge on **recall / F1 / PR-AUC**,
  not accuracy — the 84% majority makes accuracy meaningless.
- **Drop the duplicate rows** first, or at least be aware they inflate any
  metric computed with a random (non-grouped) split.
- Every feature is **self-reported survey data**, so it's noisy and the target
  is subject to recall bias — good for practice, not for clinical claims.

---
# Part 2 — Making the data training-ready

Sections 12–17 turn the raw survey table into clean, split, scaled arrays that a
model can consume directly. Each step fits **only on the training data** so no
information leaks from the test set.

## 12. Clean the data

Three cleaning steps: drop the exploration helper column, remove duplicate rows,
and clip a handful of implausible BMI values into a clinical range.

In [ ]:
# Fresh copy so the exploration above stays intact
data = df.copy()

# Remove the helper column added in Section 9 (it encodes the target -> would leak)
data = data.drop(columns=[c for c in ["is_diabetic"] if c in data.columns])

# 1) Drop exact duplicate rows
before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
print(f"Duplicates removed: {before - len(data):,} "
      f"({(before - len(data)) / before * 100:.1f}%)  ->  {len(data):,} rows left")

# 2) Confirm there is nothing to impute
print("Missing values total:", int(data.isnull().sum().sum()))

# 3) Clip implausible BMI extremes (raw max is ~98) into a sensible clinical range
n_clipped = ((data["BMI"] < 12) | (data["BMI"] > 60)).sum()
data["BMI"] = data["BMI"].clip(lower=12, upper=60)
print(f"BMI values clipped to [12, 60]: {n_clipped:,}")

## 13. Define the target

`Diabetes_012` is collapsed into a **binary** target:
`1` = prediabetes **or** diabetes (any dysglycemia), `0` = no diabetes. This
matches Kaggle's official `diabetes_binary` file and gives the model a cleaner,
less-tiny positive class than the 1.8% prediabetes group alone.

In [ ]:
# 1 = prediabetes OR diabetes, 0 = none.
# (To instead predict ONLY full diabetes, use: (data["Diabetes_012"] == 2))
data["Diabetes_binary"] = (data["Diabetes_012"] >= 1).astype(int)
data = data.drop(columns=["Diabetes_012"])

counts = data["Diabetes_binary"].value_counts().sort_index()
print(counts)
print(f"\nPositive (at-risk) rate: {data['Diabetes_binary'].mean() * 100:.1f}%")

## 14. Group the features

BRFSS columns are already numeric, but they are different *kinds* of numbers.
Binary 0/1 flags pass through untouched; the continuous and ordinal columns get
standardized so no single scale (e.g. BMI up to 60) dominates distance- or
gradient-based models.

In [ ]:
TARGET = "Diabetes_binary"

binary_cols = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
]
scale_cols = ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Age", "Education", "Income"]

# Fail loudly if a column was missed or mislabelled
missing = set(data.columns) ^ set(binary_cols + scale_cols + [TARGET])
assert not missing, f"Unclassified columns: {missing}"

print(f"Binary (passthrough): {len(binary_cols)}")
print(f"Scaled (standardized): {len(scale_cols)}")

## 15. Stratified train / test split

An 80/20 split, **stratified on the target** so both sides keep the same
positive rate — essential with imbalanced data.

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=[TARGET])
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train:", X_train.shape, "   X_test:", X_test.shape)
print("Positive rate  ->  train: {:.1f}%   test: {:.1f}%".format(
    y_train.mean() * 100, y_test.mean() * 100))

## 16. Preprocessing pipeline (fit on train only)

A `ColumnTransformer` standardizes the numeric columns and passes the binary
flags through unchanged. It is **fitted on the training set only**, then applied
to both sets — the textbook way to avoid test-set leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[("scale", StandardScaler(), scale_cols)],
    remainder="passthrough",          # binary 0/1 columns untouched
    verbose_feature_names_out=False,
)

X_train_proc = preprocessor.fit_transform(X_train)   # fit + transform on TRAIN
X_test_proc = preprocessor.transform(X_test)         # transform only on TEST

feature_names = preprocessor.get_feature_names_out()
print("Processed train matrix:", X_train_proc.shape)

# Verify the scaled columns are now ~mean 0, std 1 on the training set
check = (pd.DataFrame(X_train_proc, columns=feature_names)[scale_cols]
         .agg(["mean", "std"]).round(3))
print(check)

## 17. Save the training-ready arrays

Persist the processed arrays and the fitted preprocessor so a model script (or
the next notebook) can load them directly — no re-cleaning needed.

In [ ]:
import numpy as np
import joblib

np.savez_compressed(
    "brfss_train_test.npz",
    X_train=X_train_proc, X_test=X_test_proc,
    y_train=y_train.to_numpy(), y_test=y_test.to_numpy(),
    feature_names=np.array(feature_names, dtype=object),
)
joblib.dump(preprocessor, "brfss_preprocessor.joblib")

print("Saved:")
print("  brfss_train_test.npz       - processed X/y arrays + feature names")
print("  brfss_preprocessor.joblib  - fitted scaler (reuse on any new raw data)")

### The data is now training-ready

Load it anywhere with:

```python
import numpy as np
d = np.load("brfss_train_test.npz", allow_pickle=True)
X_train, X_test = d["X_train"], d["X_test"]
y_train, y_test = d["y_train"], d["y_test"]
```

At model time, remember:
- The positive class is **~16%** — pass `class_weight="balanced"` (or apply SMOTE
  to the *training* split only).
- Score on **recall / F1 / ROC-AUC / PR-AUC**, never plain accuracy.
- Reuse `brfss_preprocessor.joblib` to transform any new raw records the exact
  same way the model was trained on.